# Rodada confirmatória do TCC (validação → congelamento → teste → custo)

Este notebook não contém lógica do experimento: ele monta o Drive, traz o código do GitHub e chama
`scripts/colab/rodada.sh`. **A célula 1 pode ser executada quantas vezes for preciso**: cada etapa pula o que
já está feito, e a célula fica em execução mostrando o progresso (isso também evita que o Colab encerre a sessão
por inatividade).

- Ambiente: **GPU T4** (*Ambiente de execução → Alterar tipo de ambiente de execução*).
- No pedido de acesso ao Drive, use **sempre a mesma conta**: os artefatos ficam em `MyDrive/tcc-confirmatorio/`.
- O conjunto de teste só é gerado depois que `tcc-rag congelar` grava o registro de congelamento.


In [ ]:
#@title 1 · INICIAR / RETOMAR (pode rodar quantas vezes quiser)
import glob, json, os, re, subprocess, time
from google.colab import drive
from IPython.display import clear_output

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive', timeout_ms=600000)

PASTA = '/content/drive/MyDrive/tcc-confirmatorio'
RAMO = 'e06-custo-confirmatorio'
REPO = 'https://github.com/VitorEduardoLimaKenor/TCC2.git'
A = PASTA + '/artifacts'
pasta_nova = not os.path.isdir(PASTA)
os.makedirs(A, exist_ok=True)
os.makedirs('/content/logs', exist_ok=True)

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def rodando():
    return bool(sh("pgrep -f 'scripts/colab/rodada.sh'"))

gpu = sh('nvidia-smi --query-gpu=name --format=csv,noheader')
print('GPU:', gpu or 'NENHUMA  ->  Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4')
print('pasta do experimento:', PASTA, '(CRIADA AGORA: se esperava continuar uma rodada, a conta do Drive é outra)' if pasta_nova else '(já existia)')

if gpu and not rodando():
    # Depois do congelamento, o código é sempre o da revisão registrada; antes dele, o do ramo.
    congelado = A + '/congelamento.json'
    revisao = json.load(open(congelado))['codigo_revisao'] if os.path.isfile(congelado) else 'origin/' + RAMO
    if not os.path.isdir('/content/TCC2/.git'):
        sh(f'git clone -q {REPO} /content/TCC2')
    sh(f'cd /content/TCC2 && git fetch -q origin && git checkout -q --detach {revisao}')
    print('código:', sh('cd /content/TCC2 && git log --oneline -1'), '| congelado' if os.path.isfile(congelado) else '| ramo ' + RAMO)
    subprocess.Popen(f"cd /content/TCC2 && TCC_DRIVE={PASTA} nohup bash scripts/colab/rodada.sh >> /content/logs/rodada.log 2>&1 &", shell=True)
    time.sleep(3)

def contagem():
    partes = []
    for f in sorted(glob.glob(A + '/runs-v2/*.jsonl')):
        c = sh("grep -o '\"condition\": \"C[0-4]\", \"lang\": \"[a-z]*\"' '" + f + "' | sort | uniq -c | awk '{gsub(/[\",:]/,\"\"); print $3 $5 \"=\" $1}' | tr '\\n' ' '")
        partes.append(os.path.basename(f).split('-')[1] + ': ' + c)
    return partes

inicio = time.time()
while rodando():
    cauda = sh("tr '\\r' '\\n' < /content/logs/rodada.log | grep -v -E 'it/s\\]|s/it\\]|^$|Warning|warn|create_payload' | tail -8 | cut -c1-200")
    clear_output(wait=True)
    print(time.strftime('%F %T UTC', time.gmtime()), '| em execução há', int((time.time() - inicio) / 60), 'min nesta célula | GPU', sh('nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader'))
    for p in contagem():
        print('  ', p)
    print(cauda)
    sh(f"cp /content/logs/rodada.log '{PASTA}/rodada-ultima.log'")
    time.sleep(60)

clear_output(wait=True)
print(time.strftime('%F %T UTC', time.gmtime()), '| a rodada não está em execução')
for p in contagem():
    print('  ', p)
print('congelamento:', 'gravado' if os.path.isfile(A + '/congelamento.json') else 'ainda não')
print(sh("tr '\\r' '\\n' < /content/logs/rodada.log | grep -v -E 'it/s\\]|s/it\\]|^$|Warning|warn|create_payload' | tail -40 | cut -c1-220"))
sh(f"cp /content/logs/rodada.log '{PASTA}/rodada-ultima.log'")


In [ ]:
#@title 2 · EMPACOTAR E BAIXAR OS RESULTADOS (logs, análise, custo e registro de congelamento)
import os, subprocess, time
from google.colab import files
PASTA = '/content/drive/MyDrive/tcc-confirmatorio'
nome = '/content/tcc-resultados-' + time.strftime('%Y%m%d-%H%M', time.gmtime()) + '.zip'
itens = ['artifacts/runs-v2', 'artifacts/analise', 'artifacts/custo', 'artifacts/congelamento.json', 'artifacts/manifest.json',
         'artifacts/estratos_literais.json', 'artifacts/leakage_report.json', 'artifacts/db_report.json', 'artifacts/summary.json',
         'artifacts/summary_selection.json', 'artifacts/qdrant-confirmatorio.demos.identidade.json',
         'artifacts/cache/rascunhos-v2.jsonl', 'rodada-ultima.log']
existentes = [i for i in itens if os.path.exists(os.path.join(PASTA, i))]
subprocess.run(['zip', '-q', '-r', nome] + existentes, cwd=PASTA, check=True)
print(nome, round(os.path.getsize(nome) / 1e6, 2), 'MB;', len(existentes), 'itens:', existentes)
files.download(nome)
